In [ ]:

curl -fL -o hair_classifier_v1.onnx.data "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data"
curl -fL -o hair_classifier_v1.onnx "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx"
curl -fL -o sample.jpeg "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"


curl -fL -o hair_classifier_v1.onnx.data "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data"
curl -fL -o hair_classifier_v1.onnx "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx"
curl -fL -o sample.jpeg "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

sha256sum hair_classifier_v1.onnx hair_classifier_v1.onnx.data sample.jpeg

2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8 *hair_classifier_v1.onnx
6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e *hair_classifier_v1.onnx.data
6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5 *sample.jpeg

Requirement already satisfied: numpy in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 1)) (2.2.6)
Requirement already satisfied: onnxruntime in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 2)) (1.23.2)
Requirement already satisfied: Pillow in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 3)) (12.3.0)
Requirement already satisfied: coloredlogs in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 4)) (15.0.1)
Requirement already satisfied: flatbuffers in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 5)) (25.12.19)
Requirement already satisfied: humanfriendly in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 6)) (10.0)
Requirement already satisfied: packaging in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 7)) (26.3)
Requirement already satisfied: protobuf in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 8)) (7.36.2)
Requirement already satisfied: sympy in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 9)) (1.14.0)
Requirement already satisfied: mpmath in .\.conda\envs\tf_ml\lib\site-packages (from -r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 10)) (1.3.0)
Requirement already satisfied: pyreadline3 in .\.conda\envs\tf_ml\lib\site-packages (from humanfriendly->-r C:\Users\theob\Downloads\New folder\09-serverless\requirements.txt (line 6)) (3.5.6)

In [1]:

import onnxruntime as ort

session = ort.InferenceSession(
    "hair_classifier_v1.onnx",
    providers=["CPUExecutionProvider"]
)

for node in session.get_inputs():
    print("Input:", node.name, node.shape)

for node in session.get_outputs():
    print("Output:", node.name, node.shape)


Input: input ['s77', 3, 200, 200]
Output: output ['s77', 1]


In [3]:
from io import BytesIO
from urllib import request

import numpy as np
from PIL import Image


def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]

In [4]:

import numpy as np
from PIL import Image

def prepare_image(image):
    image = image.convert("RGB")
    image = image.resize((200, 200), Image.Resampling.BILINEAR)

    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )

    return np.transpose(array, (2, 0, 1))[None, ...]

image = Image.open("sample.jpeg")
tensor = prepare_image(image)

print("Original image size:", image.size)
print("Target image size:", (tensor.shape[3], tensor.shape[2]))
print("Tensor shape:", tensor.shape)


Original image size: (1024, 1024)
Target image size: (200, 200)
Tensor shape: (1, 3, 200, 200)


In [5]:

import numpy as np
from PIL import Image

def prepare_image(image):
    image = image.convert("RGB")
    image = image.resize((200, 200), Image.Resampling.BILINEAR)

    array = np.asarray(image, dtype=np.float32) / 255.0

    array = (
        array - np.array([0.485, 0.456, 0.406], dtype=np.float32)
    ) / np.array([0.229, 0.224, 0.225], dtype=np.float32)

    return np.transpose(array, (2, 0, 1))[None, ...]

image = Image.open("sample.jpeg")
tensor = prepare_image(image)

print("Tensor shape:", tensor.shape)
print("First R channel value:", tensor[0, 0, 0, 0])
print("Question 3 answer:", round(float(tensor[0, 0, 0, 0]), 3))


Tensor shape: (1, 3, 200, 200)
First R channel value: -1.0561693
Question 3 answer: -1.056


In [7]:

import numpy as np
import onnxruntime as ort
from PIL import Image

# Step 1: Prepare the image
def prepare_image(image):
    image = image.convert("RGB")
    image = image.resize(
        (200, 200), Image.Resampling.BILINEAR
    )

    array = np.asarray(image, dtype=np.float32) / 255.0

    array = (
        array - np.array(
            [0.485, 0.456, 0.406], dtype=np.float32
        )
    ) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )

    return np.transpose(array, (2, 0, 1))[None, ...]

# Step 2: Load sample image
image = Image.open("sample.jpeg")
tensor = prepare_image(image)

# Step 3: Load reference ONNX model
session = ort.InferenceSession(
    "hair_classifier_v1.onnx",
    providers=["CPUExecutionProvider"]
)

# Step 4: Run inference
output = session.run(
    ["output"],
    {"input": tensor}
)[0]

# Step 5: Display results
probability = float(output[0][0])

print("Tensor shape:", tensor.shape)
print("Raw model output:", output)
print("Straight probability:", probability)
print("Question 4 answer:", round(probability, 3))


Tensor shape: (1, 3, 200, 200)
Raw model output: [[0.72769666]]
Straight probability: 0.7276966571807861
Question 4 answer: 0.728


In [9]:
!grep "^FROM" Dockerfile

FROM public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be


In [10]:
!docker build -t mlzoomcamp-2026-serverless .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 463B 0.0s done
#1 DONE 0.1s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be
#2 DONE 3.1s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/4] FROM public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be
#4 resolve public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be 0.0s done
#4 sha256:5cead502017ed7db22e5c998a9296a3599b4b7a5fc62d8499875ceb59d1c3c72 1.58kB / 1.58kB done
#4 sha256:c51ee2d0e7c9238f8b527842e6a6d230c0009ffbf0c76761c0e5d49c57902d8b 0B / 416B 0.2s
#4 sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be 772B / 772B done
#4 sha256:bd57357d93c3fe887d45a1a1c39662e21792d42b897c1fcda65a651e6b7f7

In [11]:
!docker run --rm -p 9000:8080 mlzoomcamp-2026-serverless

^C


In [12]:

!curl -s \
  -X POST "http://localhost:9000/2015-03-31/functions/function/invocations" \
  -H "Content-Type: application/json" \
  -d '{"image_url":"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'


24 Nov 2026 18:15:51,849 [INFO] (rapid) exec '/var/runtime/bootstrap' (cwd=/var/task, handler=)
24 Nov 2026 18:37:50,354 [INFO] (rapid) Received signal signal=terminated
24 Nov 2026 18:37:50,354 [INFO] (rapid) Shutting down...
24 Nov 2026 18:37:50,354 [WARNING] (rapid) Reset initiated: SandboxTerminated
24 Nov 2026 18:37:50,354 [INFO] (rapid) Waiting for runtime domain processes termination
